# 03 - Modelisation et evaluation

Ce notebook lit les rapports generes par `06_entrainement_complet.ipynb` et commente les resultats : desequilibre, baseline, comparaison des strategies, PR-AUC, matrices de confusion et seuil optimise.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS_DIR = BASE_DIR / 'reports'
comparison = pd.read_csv(REPORTS_DIR / 'model_comparison.csv')
thresholds = pd.read_csv(REPORTS_DIR / 'threshold_analysis.csv')
baseline = pd.read_csv(REPORTS_DIR / 'baseline_analysis.csv')
comparison


,modele,strategie_desequilibre,cv_roc_auc_mean,cv_pr_auc_mean,cv_recall_mean,cv_f1_mean,cv_precision_mean,test_threshold_default,test_roc_auc,test_pr_auc,...,best_f1_recall,best_f1_precision,best_f1,best_f1_accuracy,best_f1_fp,best_f1_fn,best_f1_tp,best_f1_tn,modele_final,threshold_recommande
0,XGBoost_scale_pos_weight,scale_pos_weight,0.783544,0.262367,0.533631,0.356336,0.267516,0.5,0.796615,0.281602,...,0.696078,0.267420,0.386395,0.7745,389,62,142,1407,True,0.2
1,LogisticRegression_class_weight,class_weight_balanced,0.757199,0.249929,0.670765,0.325578,0.214961,0.5,0.750033,0.251288,...,0.328431,0.335000,0.331683,0.8650,133,137,67,1663,False,NaN
2,LogisticRegression_random_over,random_over_sampling,0.755413,0.248444,0.662187,0.321535,0.212330,0.5,0.746015,0.246112,...,0.387255,0.283154,0.327122,0.8375,200,125,79,1596,False,NaN
3,LogisticRegression_random_under,random_under_sampling,0.737246,0.235483,0.668292,0.306761,0.199080,0.5,0.739440,0.244048,...,0.617647,0.216495,0.320611,0.7330,456,78,126,1340,False,NaN
4,LogisticRegression_smote,smote,0.752987,0.249485,0.651166,0.317309,0.209787,0.5,0.741231,0.246818,...,0.544118,0.231250,0.324561,0.7690,369,93,111,1427,False,NaN
5,RandomForest_class_weight,class_weight_balanced_subsample,0.795146,0.267968,0.157873,0.204125,0.291087,0.5,0.798002,0.262276,...,0.808824,0.259027,0.392390,0.7445,472,39,165,1324,False,NaN
6,DeepLearning_smote,smote,0.645097,0.175876,0.205626,0.207364,0.209146,0.5,0.699318,0.207502,...,0.406863,0.253049,0.312030,0.8170,245,121,83,1551,False,NaN
7,LogisticRegression_baseline,aucun_reequilibrage,0.754818,0.247464,0.028146,0.051790,0.327381,0.5,0.747691,0.251169,...,0.504902,0.235698,0.321373,0.7825,334,101,103,1462,False,NaN
8,RandomForest_baseline,aucun_reequilibrage,0.803621,0.298235,0.002446,0.004875,0.666667,0.5,0.805013,0.324355,...,0.553922,0.298153,0.387650,0.8215,266,91,113,1530,False,NaN


## Baseline et accuracy trompeuse

La baseline montre pourquoi l'accuracy seule n'est pas suffisante. Un modele peut avoir une accuracy correcte en privilegiant la classe majoritaire, tout en laissant trop de faux negatifs.


In [2]:
baseline[['modele', 'strategie_desequilibre', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'tn', 'fp', 'fn', 'tp']]


,modele,strategie_desequilibre,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,NaiveMajority,baseline_majoritaire,0.8980,0.000000,0.000000,0.000000,NaN,NaN,1796,0,204,0
1,LogisticRegression_baseline,aucun_reequilibrage,0.8955,0.413793,0.058824,0.103004,0.747691,0.251169,1779,17,192,12


## Comparaison des strategies de desequilibre

Le recall reste prioritaire car l'objectif est de detecter les clients qui risquent de partir. La PR-AUC est ajoutee car elle est plus informative que la ROC-AUC quand la classe positive est rare.


In [3]:
cols = [
    'modele', 'strategie_desequilibre', 'test_recall', 'test_precision', 'test_f1',
    'test_roc_auc', 'test_pr_auc', 'test_fp', 'test_fn', 'test_tp', 'test_tn',
    'best_f1_threshold', 'best_f1'
]
comparison[cols].sort_values(['test_recall', 'test_pr_auc'], ascending=False)


,modele,strategie_desequilibre,test_recall,test_precision,test_f1,test_roc_auc,test_pr_auc,test_fp,test_fn,test_tp,test_tn,best_f1_threshold,best_f1
0,XGBoost_scale_pos_weight,scale_pos_weight,0.696078,0.267420,0.386395,0.796615,0.281602,389,62,142,1407,0.50,0.386395
1,LogisticRegression_class_weight,class_weight_balanced,0.671569,0.197406,0.305122,0.750033,0.251288,557,67,137,1239,0.75,0.331683
2,LogisticRegression_random_over,random_over_sampling,0.666667,0.199122,0.306652,0.746015,0.246112,547,68,136,1249,0.70,0.327122
3,LogisticRegression_random_under,random_under_sampling,0.661765,0.192857,0.298673,0.739440,0.244048,565,69,135,1231,0.55,0.320611
4,LogisticRegression_smote,smote,0.651961,0.190000,0.294248,0.741231,0.246818,567,71,133,1229,0.60,0.324561
5,RandomForest_class_weight,class_weight_balanced_subsample,0.377451,0.315574,0.343750,0.798002,0.262276,167,127,77,1629,0.35,0.392390
6,DeepLearning_smote,smote,0.279412,0.241525,0.259091,0.699318,0.207502,179,147,57,1617,0.20,0.312030
7,LogisticRegression_baseline,aucun_reequilibrage,0.058824,0.413793,0.103004,0.747691,0.251169,17,192,12,1779,0.15,0.321373
8,RandomForest_baseline,aucun_reequilibrage,0.000000,0.000000,0.000000,0.805013,0.324355,0,204,0,1796,0.20,0.387650


## Sensibilite du seuil

Le tableau suivant montre l'impact du seuil sur les faux positifs et faux negatifs. Un seuil plus bas augmente le recall, mais alerte plus de clients.


In [4]:
thresholds[['modele', 'strategie_desequilibre', 'threshold', 'recall', 'precision', 'f1', 'accuracy', 'fp', 'fn', 'tp', 'tn', 'clients_alertes']].head(20)


,modele,strategie_desequilibre,threshold,recall,precision,f1,accuracy,fp,fn,tp,tn,clients_alertes
0,LogisticRegression_random_over,random_over_sampling,0.10,1.000000,0.104562,0.189327,0.1265,1747,0,204,49,1951
1,LogisticRegression_class_weight,class_weight_balanced,0.10,1.000000,0.104508,0.189239,0.1260,1748,0,204,48,1952
2,LogisticRegression_random_under,random_under_sampling,0.10,1.000000,0.104401,0.189064,0.1250,1750,0,204,46,1954
3,RandomForest_class_weight,class_weight_balanced_subsample,0.10,1.000000,0.102256,0.185539,0.1045,1791,0,204,5,1995
4,LogisticRegression_random_under,random_under_sampling,0.15,0.990196,0.108544,0.195642,0.1695,1659,2,202,137,1861
5,LogisticRegression_class_weight,class_weight_balanced,0.15,0.985294,0.108707,0.195811,0.1745,1648,3,201,148,1849
6,LogisticRegression_smote,smote,0.10,0.985294,0.106181,0.191702,0.1525,1692,3,201,104,1893
7,LogisticRegression_random_over,random_over_sampling,0.15,0.975490,0.107976,0.194431,0.1755,1644,5,199,152,1843
8,RandomForest_class_weight,class_weight_balanced_subsample,0.15,0.970588,0.120511,0.214402,0.2745,1445,6,198,351,1643
9,LogisticRegression_random_under,random_under_sampling,0.20,0.965686,0.114535,0.204782,0.2350,1523,7,197,273,1720


## Modele final

Le modele final est celui qui maximise le recall avec une contrainte minimale de precision. Ce choix limite les faux negatifs tout en gardant un volume d'alertes defendable pour une equipe CRM.


In [5]:
comparison[comparison['modele_final'] == True][[
    'modele', 'strategie_desequilibre', 'threshold_recommande', 'test_recall',
    'test_precision', 'test_f1', 'test_pr_auc', 'test_fp', 'test_fn', 'test_tp', 'test_tn'
]]


,modele,strategie_desequilibre,threshold_recommande,test_recall,test_precision,test_f1,test_pr_auc,test_fp,test_fn,test_tp,test_tn
0,XGBoost_scale_pos_weight,scale_pos_weight,0.2,0.696078,0.26742,0.386395,0.281602,389,62,142,1407
